# aicode:0x — Fine-tuning Qwen2.5-Coder-1.5B

Fine-tunes **Qwen2.5-Coder-1.5B-Instruct** using QLoRA on Google Colab (free T4 GPU).
Output: a GGUF file you can import into Ollama as `aicode:0x`.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `train_data.jsonl` when prompted
3. Run all cells top to bottom
4. Download the GGUF from the output

In [ ]:
# Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable GPU in Runtime > Change runtime type')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1), 'GB' if torch.cuda.is_available() else 'N/A')

In [ ]:
!pip install -q transformers peft trl datasets bitsandbytes accelerate

In [ ]:
# Upload train_data.jsonl
from google.colab import files
print('Upload train_data.jsonl:')
uploaded = files.upload()
TRAIN_FILE = list(uploaded.keys())[0]
print(f'Using: {TRAIN_FILE}')

In [ ]:
import json, os
from datasets import Dataset

data = []
with open(TRAIN_FILE) as f:
    for line in f:
        data.append(json.loads(line.strip()))
print(f'Loaded {len(data)} examples')
dataset = Dataset.from_list(data)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
OUTPUT_DIR = './aicode0x-output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig

training_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy='epoch',
    bf16=True,
    max_seq_length=4096,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
# Save LoRA adapter
ADAPTER_DIR = os.path.join(OUTPUT_DIR, 'lora-adapter')
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Adapter saved to', ADAPTER_DIR)

In [ ]:
# Merge LoRA into base model
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='cpu',
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged = merged.merge_and_unload()

MERGED_DIR = os.path.join(OUTPUT_DIR, 'merged')
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print('Merged model saved to', MERGED_DIR)

In [ ]:
# Convert to GGUF
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git /tmp/llama.cpp
!pip install -q -r /tmp/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

!python /tmp/llama.cpp/convert_hf_to_gguf.py \
    {OUTPUT_DIR}/merged \
    --outfile {OUTPUT_DIR}/aicode0x-Q4_K_M.gguf \
    --outtype q4_k_m

print('GGUF exported:', OUTPUT_DIR + '/aicode0x-Q4_K_M.gguf')

In [ ]:
# Download the GGUF
from google.colab import files
files.download(OUTPUT_DIR + '/aicode0x-Q4_K_M.gguf')
print('Download starting...')